The contents of this directory benchmark various model approaches. Results are saved as
1. Actual models
2. Slurm logs
3. HTML Dask performance reports

In [18]:
# Imports
import subprocess
from datetime import datetime
import pickle

import statsmodels

In [29]:
#functions for model submission...

data_root="/gpfs/gibbs/pi/reilly/tabula_data"

def bench(model_code):
    """
    Executes a particular model design & collects statistics. 
    """
    now=datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    
    logdir=f"{data_root}/speed_test/logs/{model_code}_{now}"

    #make a directory to put all the log files : this will be 
    subprocess.run(f"mkdir {logdir}",shell=True)

    command=f"""
    module load miniconda
    conda activate biopython
    code_location=$(pwd)
    cd {logdir}
    python ${{code_location}}/cluster.py {model_code}
    """


    slurm_cmd = [
        "sbatch",
        "--partition=ycga",
        "--time=1:00:00",
        f"--output={logdir}/master_{model_code}_{now}.out",
        "-c 1",
        f"-J {model_code}_master",
        "--wrap", command
    ]
    result = subprocess.run(
        slurm_cmd, 
        capture_output=True, 
        text=True
    )
    
    print(result.stdout.strip() if result.returncode == 0 else result.stderr.strip())



In [30]:
bench("c900090")#fake data statsmodels multiplicative model
#bench("c900010")#fake data split by cell-type

Submitted batch job 50629603


In [12]:


def load_model(model_file:str):
    with open(f"{data_root}/speed_test/models/{model_file}","rb") as f:
        return pickle.load(f)

working_name="c900090_2025-04-21_18-40-55.pkl"
mod=load_model(working_name)


In [14]:
dir(mod)

['__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_cache',
 '_data_attr',
 '_data_in_cache',
 '_dispersion_factor',
 '_get_endog_name',
 '_get_robustcov_results',
 '_get_wald_nonlinear',
 '_transform_predict_exog',
 '_use_t',
 'aic',
 'bic',
 'bse',
 'conf_int',
 'converged',
 'cov_kwds',
 'cov_params',
 'cov_type',
 'df_model',
 'df_resid',
 'f_test',
 'fittedvalues',
 'get_diagnostic',
 'get_distribution',
 'get_influence',
 'get_margeff',
 'get_prediction',
 'im_ratio',
 'info_criteria',
 'initialize',
 'k_constant',
 'llf',
 'llnull',
 'llr',
 'llr_pvalue',
 'load',
 'method',
 'mle_retvals',
 'mle_settings',
 'model',
 'nobs',
 'normalized_cov_para

In [20]:
def check_convergence(model):
    if isinstance(model,statsmodels.discrete.count_model.ZeroInflatedNegativeBinomialResultsWrapper):
        print(model.converged)
    else:
        print("[!] Type not implemented yet.")

check_convergence(mod)

True
